# Rețele neuronale și deep learning — Seminarul 2: Antrenarea rețelelor: propagarea înapoi a erorii și optimizarea
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L2_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
import torch, torch.nn as nn
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from scipy import stats
torch.set_num_threads(1)
ca = pd.read_csv(DATA + 'california_housing.csv')
feat = [c for c in ca.columns if c != 'MedHouseVal_100k_usd']
Xc = ca[feat].values.astype(np.float32); yc = ca['MedHouseVal_100k_usd'].values.astype(np.float32)
i_tr, i_te = train_test_split(np.arange(len(ca)), test_size=0.2, random_state=SEED)
i_tr, i_va = train_test_split(i_tr, test_size=0.25, random_state=SEED)
mu, sd = Xc[i_tr].mean(0), Xc[i_tr].std(0); Z = (Xc - mu) / sd          # scaling fitted on the training rows only
Xtr, Xva, Xte, ytr, yva, yte = Z[i_tr], Z[i_va], Z[i_te], yc[i_tr], yc[i_va], yc[i_te]
print('train / validation / test:', len(i_tr), len(i_va), len(i_te))
print('linear regression, validation MSE:', round(float(np.mean((LinearRegression().fit(Xtr, ytr).predict(Xva) - yva) ** 2)), 3))

def make_mlp(d, hidden=(64, 64), act=nn.ReLU):
    layers, k = [], d
    for h in hidden:
        layers += [nn.Linear(k, h), act()]; k = h
    return nn.Sequential(*layers, nn.Linear(k, 1))

def make_opt(name, params, lr):
    return {'sgd': lambda: torch.optim.SGD(params, lr=lr), 'momentum': lambda: torch.optim.SGD(params, lr=lr, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(params, lr=lr), 'adam': lambda: torch.optim.Adam(params, lr=lr),
            'adamw': lambda: torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)}[name]()

def run_reg(opt_name, lr, seed=0, epochs=60, bs=128, sched=None):
    """MLP 64-64 on California housing; returns the validation MSE after each epoch and the final test MSE"""
    torch.manual_seed(SEED + seed); rng = np.random.default_rng(SEED + seed)
    m = make_mlp(Xtr.shape[1]); opt = make_opt(opt_name, m.parameters(), lr)
    spe = int(np.ceil(len(Xtr) / bs)); total = epochs * spe
    sch = {'cosine': lambda: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total),
           'step': lambda: torch.optim.lr_scheduler.StepLR(opt, step_size=20 * spe, gamma=0.1),
           'onecycle': lambda: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total, pct_start=0.3)}.get(sched, lambda: None)()
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr).view(-1, 1); Xv, yv = torch.tensor(Xva), torch.tensor(yva).view(-1, 1)
    val = []
    for ep in range(epochs):
        idx = rng.permutation(len(Xt))
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]; opt.zero_grad(); nn.MSELoss()(m(Xt[b]), yt[b]).backward(); opt.step()
            if sch is not None:
                sch.step()
        with torch.no_grad():
            v = float(nn.MSELoss()(m(Xv), yv)); val.append(v if np.isfinite(v) else 1e9)
    with torch.no_grad():
        te = float(nn.MSELoss()(m(torch.tensor(Xte)), torch.tensor(yte).view(-1, 1)))
    return val, te

**Sarcina 1.** Calculați manual (NumPy) gradientul MSE pentru regresia liniară $\hat y=\mathbf X\mathbf w$ pe setul de antrenare California, în $\mathbf w=\mathbf 0$, și verificați-l cu diferențierea automată din PyTorch.

In [ ]:
# codul vostru aici

**Sarcina 2.** Găsiți cea mai mare valoare proprie a hessianei $2\mathbf X^\top\mathbf X/n$ a acestei regresii și rulați coborîrea pe gradient cu $\eta=1{,}9/\lambda_{\max}$ și $\eta=2{,}1/\lambda_{\max}$ timp de 100 de pași. Ce se întîmplă?

In [ ]:
# codul vostru aici

**Sarcina 3.** Antrenați MLP 64-64 cu Adam la ratele de învățare 1e-4, 1e-3 și 1e-2 (cîte 3 inițializări, 30 de epoci). Ce rată ați păstra?

In [ ]:
# codul vostru aici

**Sarcina 4.** Comparați o rată de învățare constantă cu descreșterea cosinus pentru SGD cu moment (3 inițializări). Raportați MSE medie pe test pentru fiecare.

In [ ]:
# codul vostru aici

**Sarcina 5.** Scrieți pierderea pinball $\rho_\tau(u)=u(\tau-\mathbb 1\{u<0\})$ și verificați numeric că minimul ei după o constantă $q$, pe randamentele logaritmice zilnice S&P 500, este cuantila empirică de 1 %.

In [ ]:
# codul vostru aici